In [13]:
prompt_template = '''
You are a Jira ticket quality evaluator.

Evaluate the following ticket based on these criteria:
1. Problem clarity
2. Reproduction steps
3. Environment details
4. Impact
5. Evidence (logs/screenshots)

For each criterion:
- Mark: PRESENT / PARTIAL / MISSING
- Give a short reason

Then:
- Assign a score out of 8
- Classify as:
  COMPLETE (7–8)
  PARTIAL (4–6)
  INCOMPLETE (0–3)

Finally:
- List missing information
- Suggest what user should add

Return response in JSON format:

{{
  "score": number,
  "classification": "..."
}}

Ticket:
{ticket_data}
'''

In [19]:
import os
from dotenv import load_dotenv
load_dotenv()

True

In [16]:

from langchain_groq import ChatGroq

llm = ChatGroq(
    model="openai/gpt-oss-120b",  # or llama3-8b-8192
    temperature=0
)

In [4]:
jira_ticket_text = '''
    Title: Login failing for some users

Description:
Some users are unable to login to the system.

Steps to Reproduce:
1. Go to login page
2. Enter credentials
3. Click login

Expected Behavior:
User should be logged in.

Actual Behavior:
Login fails with error message "Something went wrong"

Environment:
- Environment: Production

'''

In [14]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_template(prompt_template)

print(prompt)



input_variables=['ticket_data'] input_types={} partial_variables={} messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['ticket_data'], input_types={}, partial_variables={}, template='\nYou are a Jira ticket quality evaluator.\n\nEvaluate the following ticket based on these criteria:\n1. Problem clarity\n2. Reproduction steps\n3. Environment details\n4. Impact\n5. Evidence (logs/screenshots)\n\nFor each criterion:\n- Mark: PRESENT / PARTIAL / MISSING\n- Give a short reason\n\nThen:\n- Assign a score out of 8\n- Classify as:\n  COMPLETE (7–8)\n  PARTIAL (4–6)\n  INCOMPLETE (0–3)\n\nFinally:\n- List missing information\n- Suggest what user should add\n\nReturn response in JSON format:\n\n{{\n  "score": number,\n  "classification": "..."\n}}\n\nTicket:\n{ticket_data}\n'), additional_kwargs={})]


In [17]:
chain = prompt | llm

response = chain.invoke({
    "ticket_data": jira_ticket_text
})

response

AIMessage(content='{\n  "score": 2,\n  "classification": "INCOMPLETE",\n  "missing_information": [\n    "Specific users affected (e.g., usernames, roles, or IDs)",\n    "Detailed error information (full error message, stack trace, error codes)",\n    "Browser, OS, and version details",\n    "Severity and business impact (how many users, any downtime, SLA breach)",\n    "Logs, screenshots, or any other evidence showing the failure"\n  ],\n  "suggestions": [\n    "Add a list of example accounts that cannot log in, or describe the pattern (e.g., all users from a certain domain).",\n    "Include the exact error message text, any error codes, and a screenshot of the failure dialog.",\n    "Specify the browsers and operating systems used when reproducing the issue, along with their versions.",\n    "Explain the impact: is it affecting all users, a specific group, or causing critical downtime?",\n    "Attach relevant logs from the server or client, or provide a link to a log file."\n  ]\n}', 

# Jira setup

In [18]:
pip install atlassian-python-api



   ---------------------------------------- 0/3 [jmespath]
   -------------------------- ------------- 2/3 [atlassian-python-api]
   -------------------------- ------------- 2/3 [atlassian-python-api]
   -------------------------- ------------- 2/3 [atlassian-python-api]
   -------------------------- ------------- 2/3 [atlassian-python-api]
   -------------------------- ------------- 2/3 [atlassian-python-api]
   ---------------------------------------- 3/3 [atlassian-python-api]




[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: C:\Users\Vansh Rupansh\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.10_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [25]:
pip install jira

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: C:\Users\Vansh Rupansh\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.10_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [40]:
JIRA_URL="https://aiteam-test.atlassian.net"
JIRA_EMAIL="vanshkapoorvk7@gmail.com"
JIRA_PAT=os.getenv("JIRA_TOKEN")

In [30]:
from jira import JIRA

jira = JIRA(
    server=JIRA_URL,
    basic_auth=(JIRA_EMAIL, JIRA_PAT)
)

issues = jira.search_issues("project=Triage AND status='To Do'")

In [31]:
issues

[<JIRA Issue: key='SCRUM-4', id='10003'>,
 <JIRA Issue: key='SCRUM-1', id='10000'>]

In [57]:
issue = jira.issue("SCRUM-5")

In [58]:
print(issue.key)  # MYPROJ-123
print(issue.fields.summary)
print(issue.fields.description)

SCRUM-5
Build a jira triage system
Tasks:

* setup Jira connection
* create llm
* build a prompt
* analyse ticket using llm and generate a score


Expertise: AI, LLM, langchain


In [37]:
print(type(issue.fields.description))

<class 'str'>


In [59]:
ticket_text = issue.fields.description

In [53]:
chain = prompt | llm

response = chain.invoke({
    "ticket_data": issue.fields.description
})

response.content

'{\n  "score": 0,\n  "classification": "INCOMPLETE",\n  "missing_information": [\n    "Problem clarity – no description of the issue or goal beyond a vague task list.",\n    "Reproduction steps – no steps to reproduce the problem or to verify the tasks.",\n    "Environment details – no information about Jira version, OS, LLM platform, etc.",\n    "Impact – no indication of who is affected, severity, or business impact.",\n    "Evidence – no logs, screenshots, or other supporting material."\n  ],\n  "suggestions": [\n    "Add a clear problem statement explaining what is not working or what needs to be achieved.",\n    "Provide step‑by‑step instructions to reproduce the issue or to complete the tasks.",\n    "Include environment details such as Jira version, LLM framework, OS, browser, and any relevant configuration.",\n    "Describe the impact: who is affected, how critical it is, and any deadlines.",\n    "Attach relevant evidence like error logs, console output, or screenshots that il

In [39]:
response.content

'{\n  "score": 0,\n  "classification": "INCOMPLETE",\n  "missing_information": [\n    "Problem clarity – no description of the issue or goal beyond a vague task list.",\n    "Reproduction steps – no steps provided to reproduce the problem or to verify the tasks.",\n    "Environment details – no information about Jira version, OS, LLM platform, network setup, etc.",\n    "Impact – no indication of who/what is affected, urgency, or business impact.",\n    "Evidence – no logs, screenshots, error messages, or any supporting artifacts."\n  ],\n  "suggestions": [\n    "Add a clear problem statement: what is not working or what needs to be achieved.",\n    "Provide step‑by‑step reproduction instructions so anyone can follow the same process.",\n    "Include environment specifics: Jira instance/version, LLM framework, OS, browser, network constraints, etc.",\n    "Explain the impact: who is blocked, how it affects the workflow, and any deadlines.",\n    "Attach relevant evidence such as error 

## Confluence connection

## User Phonebook Test

In [42]:
CONFLUENCE_URL="https://aiteam-test.atlassian.net"

In [43]:
from atlassian import Confluence

confluence = Confluence(
    url=CONFLUENCE_URL,
    username=JIRA_EMAIL,
    password=JIRA_PAT
)

In [44]:
page = confluence.get_page_by_id(
    page_id="589825",
    expand="body.storage"
)

print(page["title"])
print(page["body"]["storage"]["value"])

Team phonebook
<table data-table-width="760" data-layout="default" ac:local-id="aa4968a8b709"><tbody><tr ac:local-id="55f87022d7ce"><th ac:local-id="c8b614c980a6"><p local-id="cd295630fd00"><strong>users</strong></p></th><th ac:local-id="77deef52b8cf"><p local-id="ceb6e3d031bf"><strong>expertise</strong></p></th><th ac:local-id="17ed43f629aa"><p local-id="cdcb11ff8998" /></th></tr><tr ac:local-id="c48d41d38c87"><td ac:local-id="e7a85cd60c78"><p local-id="caa4a47f952d">Vansh</p></td><td ac:local-id="7f5f25115b86"><p local-id="b14520e2f8f5">annotation component, langchain, LLM</p></td><td ac:local-id="6afbc4a607f9"><p local-id="9ebbffbe54e5" /></td></tr><tr ac:local-id="3f9e642877c5"><td ac:local-id="1bddf2c0ef2a"><p local-id="a53300a41d5e">Deepesh</p></td><td ac:local-id="060507c4d823"><p local-id="78d25e758274">qmetry, E2E automation</p></td><td ac:local-id="a6716429159c"><p local-id="49a020f0bc1b" /></td></tr><tr ac:local-id="c6b2c8388e25"><td ac:local-id="f9e1dfa2014c"><p local-id="2

### Convert html table to clean text

In [45]:
from bs4 import BeautifulSoup

def html_to_text(html):
    soup = BeautifulSoup(html, "html.parser")
    return soup.get_text(separator="\n")

content = html_to_text(page["body"]["storage"]["value"])

print(content)

users
expertise
Vansh
annotation component, langchain, LLM
Deepesh
qmetry, E2E automation
Anuj
Team planning
Chakri Sai
Unit Testing, tiff
Sree Harsha
patient service backend


In [46]:
def format_confluence_page(page):
    html = page["body"]["storage"]["value"]
    text = html_to_text(html)

    return f"""
Title: {page['title']}

Content:
{text}
"""

content = format_confluence_page(page)

print(content)


Title: Team phonebook

Content:
users
expertise
Vansh
annotation component, langchain, LLM
Deepesh
qmetry, E2E automation
Anuj
Team planning
Chakri Sai
Unit Testing, tiff
Sree Harsha
patient service backend



In [47]:
from bs4 import BeautifulSoup

# html = """<your_table_html_here>"""

def parse_table(html):
    soup = BeautifulSoup(html, "html.parser")
    
    table = soup.find("table")
    rows = table.find_all("tr")
    
    data = []
    
    # Extract headers
    headers = [
        th.get_text(strip=True).lower()
        for th in rows[0].find_all("th")
    ]
    
    # Extract rows
    for row in rows[1:]:
        cols = row.find_all("td")
        
        if not cols:
            continue
        
        row_data = {}
        
        for i, col in enumerate(cols):
            text = col.get_text(strip=True)
            
            if i < len(headers) and headers[i]:
                row_data[headers[i]] = text
        
        if row_data:
            data.append(row_data)
    
    return data


def normalize(data):
    for row in data:
        if "expertise" in row:
            row["expertise"] = [
                e.strip() for e in row["expertise"].split(",")
            ]
    return data


def getHTMLTableToJson(data):
    table_data = parse_table(data)
    normalize_data = normalize(table_data)
    return normalize_data

In [51]:
users = getHTMLTableToJson(page["body"]["storage"]["value"])
users

[{'users': 'Vansh', 'expertise': ['annotation component', 'langchain', 'LLM']},
 {'users': 'Deepesh', 'expertise': ['qmetry', 'E2E automation']},
 {'users': 'Anuj', 'expertise': ['Team planning']},
 {'users': 'Chakri Sai', 'expertise': ['Unit Testing', 'tiff']},
 {'users': 'Sree Harsha', 'expertise': ['patient service backend']}]

## Run LLM

In [61]:
prompt_template = """
You are an intelligent task assignment system.

Your job is to assign the best user for a Jira ticket based on expertise match.

Rules:
- Match ticket requirements with user expertise
- Prefer users with closest and most relevant expertise
- If multiple users match, choose the best fit
- If no strong match, return "NO_MATCH"

Return ONLY JSON:

{{
  "assigned_user": "...",
  "reason": "...",
  "confidence": 0-1
}}

Users:
{users_json}

Ticket:
{ticket_data}
"""

In [62]:
from langchain_core.prompts import ChatPromptTemplate
import json

prompt = ChatPromptTemplate.from_template(prompt_template)

chain = prompt | llm


In [63]:
response = chain.invoke({
    "users_json": json.dumps(users, indent=2),
    "ticket_data": ticket_text
})

response

AIMessage(content='{\n  "assigned_user": "Vansh",\n  "reason": "Vansh\'s expertise includes both \'langchain\' and \'LLM\', directly matching the ticket\'s requirements for AI, LLM, and langchain tasks.",\n  "confidence": 0.95\n}', additional_kwargs={'reasoning_content': 'We need to assign best user based on expertise. Ticket requires AI, LLM, langchain. Users:\n\n- Vansh: annotation component, langchain, LLM -> matches langchain and LLM.\n- Deepesh: qmetry, E2E automation -> no.\n- Anuj: Team planning -> no.\n- Chakri Sai: Unit Testing, tiff -> no.\n- Sree Harsha: patient service backend -> no.\n\nThus assign Vansh. Provide reason and confidence. Confidence high, maybe 0.95.\n\nReturn JSON only.'}, response_metadata={'token_usage': {'completion_tokens': 182, 'prompt_tokens': 365, 'total_tokens': 547, 'completion_time': 0.400760885, 'completion_tokens_details': {'reasoning_tokens': 114}, 'prompt_time': 0.019291171, 'prompt_tokens_details': None, 'queue_time': 0.055332051, 'total_time':

In [64]:
response.content

'{\n  "assigned_user": "Vansh",\n  "reason": "Vansh\'s expertise includes both \'langchain\' and \'LLM\', directly matching the ticket\'s requirements for AI, LLM, and langchain tasks.",\n  "confidence": 0.95\n}'